In [ ]:
# ── 1: Imports ───────────────────────────────────────────
import sys
sys.path.append('C:/Users/Ready2Use/Documents/my-folder/Ironhack-week8/river-risk-index/Scr')
 
import os
from dotenv import load_dotenv
from cleaning_functions import load_config
from q1_functions import *

In [ ]:
# ── 2: Connect ───────────────────────────────────────────
load_dotenv()
config = load_config()
engine = get_engine(os.getenv("DB_PASSWORD"))

In [ ]:
# ── 3: Build countries master table ─────────────────────
df7 = pd.read_parquet(config["output_data"]["file7"])
df1 = pd.read_parquet(config["output_data"]["file1"])
 
countries_db = build_countries_master(df7, df1)
print(countries_db.shape)
countries_db.head()

In [ ]:
# ── 4: Upload continents to SQL ─────────────────────────
continents = pd.DataFrame({
    'continent_name': ['Africa', 'Asia', 'Europe', 'North America',
                       'South America', 'Oceania', 'Antarctica', 'Unknown']
})
continents.to_sql('continents', engine, if_exists='append', index=False)
print("✅ continents loaded")

In [ ]:
# ── 5: Upload countries to SQL ──────────────────────────
countries_db.rename(columns={'country_name': 'country_name'}).to_sql(
    'countries', engine, if_exists='append', index=False
)
print("✅ countries loaded")

In [ ]:
# ── 6: Build and upload plastic_generation ───────────────
df6  = pd.read_parquet(config["output_data"]["file6"])
pg_db = build_plastic_generation_db(df6, countries_db)
 
pg_db['country_id'] = pg_db['country_id'].astype(int)
pg_db.to_sql('plastic_generation', engine, if_exists='append', index=False)
print("✅ plastic_generation loaded")

In [ ]:
# ── 7: Build and upload emission_points ──────────────────
df_countries = pd.read_sql(
    "SELECT country_id, country_name, continent_id, income_group FROM countries",
    engine
)
 
rivers_full   = pd.read_parquet(config["output_data"]["file1"])
emission_points = build_emission_points(rivers_full, df_countries)
 
emission_points.to_sql(
    'emission_points', engine, if_exists='replace',
    index=True, index_label='point_id'
)
print(f"✅ emission_points loaded: {len(emission_points)} rows")

In [ ]:
# ── 8: Plot — emission distribution ──────────────────────
plot_emission_distribution(engine)

In [ ]:
# ── 9: Plot — emissions by income group ──────────────────
plot_emissions_by_income_group(engine)
 

In [ ]:
# ── 10: Plot — choropleth map ────────────────────────────
plot_choropleth(engine)